In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import gradio as gr
from tqdm import tqdm
import time

# ==========================================
# 1. СРЕДА (без изменений)
# ==========================================
class TicTacToe7x7:
    def __init__(self):
        self.board = np.zeros(49, dtype=np.int8)
        self.current_player = 1

    def reset(self):
        self.board = np.zeros(49, dtype=np.int8)
        self.current_player = 1
        return self.get_relative_state()

    def get_relative_state(self):
        return self.board * self.current_player

    def get_valid_actions(self):
        return np.where(self.board == 0)[0]

    def check_win(self, player):
        b = self.board.reshape(7, 7)
        for r in range(7):
            for c in range(4):
                if np.all(b[r, c:c+4] == player): return True
        for r in range(4):
            for c in range(7):
                if np.all(b[r:r+4, c] == player): return True
        for r in range(4):
            for c in range(4):
                if np.all(np.diag(b[r:r+4, c:c+4]) == player): return True
        for r in range(4):
            for c in range(3, 7):
                if np.all(np.diag(np.fliplr(b[r:r+4, c-3:c+1])) == player): return True
        return False

    def step(self, action):
        self.board[action] = self.current_player
        done = False
        reward = 0

        if self.check_win(self.current_player):
            reward = 1
            done = True
        elif len(self.get_valid_actions()) == 0:
            reward = 0
            done = True

        next_state = self.get_relative_state()
        self.current_player *= -1
        
        return next_state, reward, done

# ==========================================
# 2. CNN АГЕНТ (УЛУЧШЕННЫЙ)
# ==========================================
class CNNQNetwork(nn.Module):
    def __init__(self):
        super(CNNQNetwork, self).__init__()
        # Вход: 1 канал (доска), 7x7
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, 49)

    def forward(self, x):
        # x shape: (batch, 49) -> (batch, 1, 7, 7)
        x = x.view(-1, 1, 7, 7)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(-1, 64 * 7 * 7)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

class DQNAgent:
    def __init__(self):
        self.network = CNNQNetwork()
        self.target_network = CNNQNetwork()
        self.target_network.load_state_dict(self.network.state_dict())
        self.optimizer = optim.Adam(self.network.parameters(), lr=0.0005)
        self.memory = ReplayBuffer(50000)
        self.gamma = 0.99

    def select_action(self, state, valid_actions, epsilon):
        if np.random.rand() < epsilon:
            return np.random.choice(valid_actions)

        state_t = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q_values = self.network(state_t).squeeze(0).numpy()

        mask = np.full(49, -np.inf)
        mask[valid_actions] = 0
        q_values += mask

        return np.argmax(q_values)

    def update(self, batch_size=128):
        if len(self.memory) < batch_size:
            return

        state, action, reward, next_state, done = self.memory.sample(batch_size)

        state_t = torch.FloatTensor(state)
        next_state_t = torch.FloatTensor(next_state)
        action_t = torch.LongTensor(action)
        reward_t = torch.FloatTensor(reward)
        done_t = torch.FloatTensor(done)

        q_values = self.network(state_t).gather(1, action_t.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_q_values = self.target_network(next_state_t).max(1)[0]
            target_q_values = reward_t + (self.gamma * next_q_values * (1 - done_t))

        loss = nn.MSELoss()(q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target_network(self):
        self.target_network.load_state_dict(self.network.state_dict())

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)

# ==========================================
# 3. ОБУЧЕНИЕ (больше эпизодов)
# ==========================================
def train_agent(episodes=20000):
    env = TicTacToe7x7()
    agent = DQNAgent()
    
    epsilon = 1.0
    epsilon_decay = 0.9995
    epsilon_min = 0.01
    
    recent_lengths = []
    recent_rewards = []
    stats_window = 100
    
    print(f"Начинаем обучение CNN агента на {episodes} эпизодах...")
    start_time = time.time()
    
    pbar = tqdm(range(1, episodes + 1), desc="Обучение", 
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')
    
    for ep in pbar:
        state = env.reset()
        done = False
        steps = 0
        step_counter = 0  # Счетчик для обновления сети
        first_player_reward = 0
        
        while not done:
            valid_actions = env.get_valid_actions()
            action = agent.select_action(state, valid_actions, epsilon)
            
            next_state, reward, done = env.step(action)
            agent.memory.push(state, action, reward, next_state, float(done))
            
            # КЛЮЧЕВОЕ ИЗМЕНЕНИЕ: обновляем сеть каждые 10 шагов
            step_counter += 1
            if step_counter % 10 == 0 and len(agent.memory) >= 256:
                agent.update(batch_size=128)  # Вернули batch size к 128
            
            if env.current_player == -1:
                first_player_reward = reward
            
            state = next_state
            steps += 1

        if ep % 10 == 0:
            agent.update_target_network()

        if epsilon > epsilon_min:
            epsilon *= epsilon_decay

        recent_lengths.append(steps)
        recent_rewards.append(first_player_reward)
        if len(recent_lengths) > stats_window:
            recent_lengths.pop(0)
            recent_rewards.pop(0)

        if ep % 50 == 0:
            avg_len = np.mean(recent_lengths)
            avg_reward = np.mean(recent_rewards)
            elapsed = time.time() - start_time
            eps_per_sec = ep / elapsed
            
            pbar.set_description(
                f"ε={epsilon:.3f} | "
                f"длина={avg_len:.1f} | "
                f"награда={avg_reward:+.2f} | "
                f"{eps_per_sec:.0f} эп/с"
            )

    pbar.close()
    
    total_time = time.time() - start_time
    print(f"\n✅ Обучение завершено за {total_time/60:.1f} минут")
    print(f"📊 Финальная статистика (последние {stats_window} игр):")
    print(f"   Средняя длина игры: {np.mean(recent_lengths):.1f} ходов")
    print(f"   Средняя награда: {np.mean(recent_rewards):+.3f}")
    
    return agent

# ==========================================
# 4. GRADIO ИНТЕРФЕЙС
# ==========================================
class GameSession:
    def __init__(self):
        self.env = TicTacToe7x7()
        self.agent = None
        self.state = self.env.reset()
        self.game_over = False

    def reset_game(self):
        self.env.reset()
        self.state = self.env.reset()
        self.game_over = False
        return self.render_board(), "Игра начата! Твой ход (крестики)."

    def render_board(self):
        board_str = ""
        for i in range(7):
            for j in range(7):
                cell = self.env.board[i * 7 + j]
                if cell == 1:
                    board_str += "❌ "
                elif cell == -1:
                    board_str += "⭕ "
                else:
                    board_str += "⬜ "
            board_str += "\n"
        return board_str

    def make_move(self, action):
        if self.game_over:
            return self.render_board(), "Игра окончена. Нажми 'Новая игра'."

        if action < 0 or action >= 49 or self.env.board[action] != 0:
            return self.render_board(), "Недопустимый ход! Выбери пустую клетку."

        # Ход игрока
        next_state, reward, done = self.env.step(action)
        self.state = next_state

        if done:
            self.game_over = True
            return self.render_board(), "🎉 Ты выиграл!"

        # Ход агента
        valid_actions = self.env.get_valid_actions()
        agent_action = self.agent.select_action(self.state, valid_actions, epsilon=0.0)
        next_state, reward, done = self.env.step(agent_action)
        self.state = next_state

        if done:
            self.game_over = True
            if reward == 1:
                return self.render_board(), "😢 Агент выиграл!"
            else:
                return self.render_board(), "🤝 Ничья!"

        return self.render_board(), f"Агент поставил в клетку {agent_action}. Твой ход."

# Глобальная сессия
session = GameSession()

def create_gradio_interface(agent):
    session.agent = agent

    def handle_click(evt: gr.SelectData):
        action = evt.index
        return session.make_move(action)

    def reset_game():
        return session.reset_game()

    with gr.Blocks(title="Крестики-нолики 7x7") as demo:
        gr.Markdown("# 🎮 Крестики-нолики 7x7 против CNN агента")
        gr.Markdown("Собери 4 в ряд, чтобы победить!")
        
        board_display = gr.Markdown(session.render_board())
        status = gr.Textbox(value="Твой ход (крестики)", label="Статус")
        
        # Сетка 7x7 с кликабельными кнопками
        with gr.Row():
            for i in range(7):
                with gr.Column():
                    for j in range(7):
                        btn = gr.Button(
                            value="⬜",
                            elem_id=f"btn_{i*7+j}",
                            size="sm"
                        )
        
        gr.Markdown("### Управление")
        action_input = gr.Number(label="Номер клетки (0-48)", value=0)
        move_btn = gr.Button("Сделать ход")
        reset_btn = gr.Button("Новая игра")
        
        # Обработчики
        move_btn.click(
            fn=lambda x: session.make_move(int(x)),
            inputs=[action_input],
            outputs=[board_display, status]
        )
        
        reset_btn.click(
            fn=reset_game,
            outputs=[board_display, status]
        )

    return demo

# ==========================================
# 5. ЗАПУСК
# ==========================================
if __name__ == "__main__":
    # Обучаем агента
    trained_agent = train_agent(episodes=20000)
    
    # Запускаем Gradio интерфейс
    demo = create_gradio_interface(trained_agent)
    demo.launch()

Начинаем обучение CNN агента на 20000 эпизодах...


ε=0.010 | длина=32.4 | награда=+0.13 | 8 эп/с: 100%|██████████| 20000/20000 [42:25<00:00] 



✅ Обучение завершено за 42.4 минут
📊 Финальная статистика (последние 100 игр):
   Средняя длина игры: 32.4 ходов
   Средняя награда: +0.130
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
